In [1]:
from google.cloud import bigquery
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

In [2]:
from google.colab import auth
auth.authenticate_user()

In [3]:
# Add the name of the project in google console
client = bigquery.Client(project="citric-trees-489611-k5")

# Data Preparation - ICU from ED Transfers

In [4]:
# Load the data
icu_from_ed_query = """
WITH ed_admissions AS (
    -- identify hospital admissions that came through the ED
    SELECT
        hadm_id,
        subject_id,
        admittime,
        admission_type,
        admission_location
    FROM physionet-data.mimiciv_3_1_hosp.admissions
    WHERE admission_location IN (
        'EMERGENCY ROOM',
        'TRANSFER FROM EMERGENCY ROOM'
    )
),

icu_stays AS (
    -- get first ICU stay per hospital admission
    SELECT
        subject_id,
        hadm_id,
        stay_id,
        first_careunit,
        intime  AS icu_intime,
        outtime AS icu_outtime,
        los     AS icu_los_days,
        ROW_NUMBER() OVER (
            PARTITION BY hadm_id
            ORDER BY intime ASC
        ) AS stay_rank
    FROM physionet-data.mimiciv_3_1_icu.icustays
),

ed_transfers AS (
    -- confirm the transfer pathway went through the ED
    SELECT
        t.hadm_id,
        t.subject_id,
        MIN(t.intime)  AS ed_intime,
        MAX(t.outtime) AS ed_outtime
    FROM physionet-data.mimiciv_3_1_hosp.transfers t
    WHERE t.careunit IN (
        'Emergency Department',
        'Emergency Department Observation'
    )
    GROUP BY t.hadm_id, t.subject_id
),

clinician_verified AS (
    -- confirm a clinician actually charted on this patient
    -- during their ICU stay window
    -- at least 2 chart events required to avoid
    -- single accidental entries
    SELECT
        ce.subject_id,
        ce.stay_id,
        COUNT(*)                    AS n_chart_events,
        COUNT(DISTINCT ce.caregiver_id) AS n_unique_caregivers,
        MIN(ce.charttime)           AS first_chart_time,
        MAX(ce.charttime)           AS last_chart_time
    FROM physionet-data.mimiciv_3_1_icu.chartevents ce
    INNER JOIN physionet-data.mimiciv_3_1_icu.icustays i
        ON ce.stay_id = i.stay_id
    WHERE ce.charttime BETWEEN i.intime AND i.outtime
    GROUP BY ce.subject_id, ce.stay_id
    HAVING COUNT(*) >= 2                -- at least 2 charted events
       AND COUNT(DISTINCT ce.caregiver_id) >= 1  -- by at least 1 caregiver
)

SELECT
    i.subject_id,
    i.hadm_id,
    i.stay_id,
    i.first_careunit,
    i.icu_intime,
    i.icu_outtime,
    i.icu_los_days,
    et.ed_intime,
    et.ed_outtime,
    TIMESTAMP_DIFF(i.icu_intime, et.ed_outtime, HOUR) AS hrs_ed_to_icu,
    ea.admission_type,
    ea.admission_location,
    cv.n_chart_events,
    cv.n_unique_caregivers,
    cv.first_chart_time,
    cv.last_chart_time
FROM icu_stays i
INNER JOIN ed_admissions ea
    ON i.hadm_id = ea.hadm_id
INNER JOIN ed_transfers et
    ON i.hadm_id = et.hadm_id
INNER JOIN clinician_verified cv           -- INNER JOIN drops patients
    ON i.stay_id = cv.stay_id             -- with no clinical contact
WHERE i.stay_rank = 1                      -- first ICU stay only
  AND i.icu_intime > et.ed_intime          -- ICU came after ED
  AND TIMESTAMP_DIFF(
        i.icu_intime, et.ed_outtime, HOUR
      ) <= 24                              -- within 24 hours of leaving ED
ORDER BY i.subject_id
"""

df_icu = client.query(icu_from_ed_query).to_dataframe()

print(df_icu.shape)
print(df_icu.dtypes)
print(df_icu.isnull().sum())

(29733, 16)
subject_id                      Int64
hadm_id                         Int64
stay_id                         Int64
first_careunit                 object
icu_intime             datetime64[us]
icu_outtime            datetime64[us]
icu_los_days                  float64
ed_intime              datetime64[us]
ed_outtime             datetime64[us]
hrs_ed_to_icu                   Int64
admission_type                 object
admission_location             object
n_chart_events                  Int64
n_unique_caregivers             Int64
first_chart_time       datetime64[us]
last_chart_time        datetime64[us]
dtype: object
subject_id             0
hadm_id                0
stay_id                0
first_careunit         0
icu_intime             0
icu_outtime            0
icu_los_days           0
ed_intime              0
ed_outtime             0
hrs_ed_to_icu          0
admission_type         0
admission_location     0
n_chart_events         0
n_unique_caregivers    0
first_chart_time

In [5]:
# Cast and parse timestamps
ts_cols = ["icu_intime", "icu_outtime", "ed_intime", "ed_outtime",
           "first_chart_time", "last_chart_time"]

for col in ts_cols:
    df_icu[col] = pd.to_datetime(df_icu[col], utc=True)

"""
Temporal context features derived from ED arrival
Hour-of-day and day-of-week capture shift patterns, staffing levels,
and bed availability — all clinically meaningful for transfer lag
"""
df_icu["ed_arrival_hour"]    = df_icu["ed_intime"].dt.hour
df_icu["ed_arrival_dow"]     = df_icu["ed_intime"].dt.dayofweek   # 0=Mon, 6=Sun
df_icu["ed_arrival_weekend"] = (df_icu["ed_arrival_dow"] >= 5).astype(int)

In [6]:
# Validate and clean durations
# hrs_ed_to_icu: drop negative values — these represent broken transfer pathway
# reconstructions (median -70 hrs, range -601 to -1), not documentation lag.
# At 0.5% of the dataset the loss is negligible.
print(f"Rows before: {len(df_icu)}")
neg_mask = df_icu["hrs_ed_to_icu"] < 0
df_icu = df_icu[~neg_mask].copy()
print(f"Rows after:  {len(df_icu)}")
print(f"Dropped:     {neg_mask.sum()} rows")

assert (df_icu["hrs_ed_to_icu"] >= 0).all()
assert (df_icu["hrs_ed_to_icu"] <= 24).all(), "Transfer times exceed 24h cap"

# icu_los_days: zero/fractional values are clinically valid (rapid deterioration,
# early death, or fast transfer) — retain them, just confirm no negatives
neg_los = df_icu[df_icu["icu_los_days"] < 0]
if not neg_los.empty:
    print(f"Warning: {len(neg_los)} rows with negative icu_los_days — review these")
    df_icu = df_icu[df_icu["icu_los_days"] >= 0].copy()

Rows before: 29733
Rows after:  29585
Dropped:     148 rows


In [7]:
# Encode categoricals
# first_careunit: nominal — which ICU type the patient landed in
careunit_dummies = pd.get_dummies(df_icu["first_careunit"], prefix="careunit")
df_icu = pd.concat([df_icu, careunit_dummies], axis=1)

# admission_type: nominal categories in MIMIC (URGENT, ELECTIVE, etc.)
admtype_dummies = pd.get_dummies(df_icu["admission_type"], prefix="adm_type")
df_icu = pd.concat([df_icu, admtype_dummies], axis=1)

In [8]:
# Derive acuity signal features
# Chart rate: events per hour of ICU stay — captures care intensity
df_icu["icu_los_hrs"] = df_icu["icu_los_days"] * 24
df_icu["icu_los_hrs"] = df_icu["icu_los_hrs"].replace(0, 0.5)   # avoid div/0 for very short stays

df_icu["chart_rate_per_hr"] = df_icu["n_chart_events"] / df_icu["icu_los_hrs"]

# Charting span: hours between first and last chart event
# Wide span = sustained monitoring; narrow span = concentrated early activity
df_icu["chart_span_hrs"] = (
    (df_icu["last_chart_time"] - df_icu["first_chart_time"])
    .dt.total_seconds() / 3600
)

# Caregiver ratio: unique caregivers per chart event
# High ratio = many hands involved = higher acuity or complex case
df_icu["caregiver_ratio"] = df_icu["n_unique_caregivers"] / df_icu["n_chart_events"]

In [9]:
# Normalize continuous features
scaler = MinMaxScaler()

continuous_cols = [
    "hrs_ed_to_icu",
    "icu_los_days",
    "n_chart_events",
    "n_unique_caregivers",
    "chart_rate_per_hr",
    "chart_span_hrs",
    "caregiver_ratio",
    "ed_arrival_hour",
]

norm_col_names = [f"{c}_norm" for c in continuous_cols]
df_icu[norm_col_names] = scaler.fit_transform(df_icu[continuous_cols])

In [10]:
# Final table
# Retain all three join keys; drop raw timestamps before handoff
# admission_location retained for audit/stratification but carries no
# discriminating model signal — every row is ED by construction
id_cols      = ["subject_id", "hadm_id", "stay_id"]
raw_features = [
    "first_careunit", "admission_type", "admission_location",
    "hrs_ed_to_icu", "icu_los_days",
    "n_chart_events", "n_unique_caregivers",
    "chart_rate_per_hr", "chart_span_hrs", "caregiver_ratio",
    "ed_arrival_hour", "ed_arrival_dow", "ed_arrival_weekend",
]
derived_norm = norm_col_names
encoded_cats = (
    [c for c in df_icu.columns if c.startswith("careunit_")] +
    [c for c in df_icu.columns if c.startswith("adm_type_")]
)

df_icu_final = df_icu[id_cols + raw_features + derived_norm + encoded_cats].copy()

print(df_icu_final.shape)
print(df_icu_final.head())

(29585, 39)
   subject_id   hadm_id   stay_id  \
0    10000032  29079034  39553978   
1    10000690  25860671  37081114   
2    10000980  26913865  39765666   
3    10002155  28994087  31090461   
4    10002155  20345487  32358465   

                                     first_careunit admission_type  \
0                Medical Intensive Care Unit (MICU)       EW EMER.   
1                Medical Intensive Care Unit (MICU)       EW EMER.   
2                Medical Intensive Care Unit (MICU)       EW EMER.   
3  Medical/Surgical Intensive Care Unit (MICU/SICU)       EW EMER.   
4                Medical Intensive Care Unit (MICU)       EW EMER.   

  admission_location  hrs_ed_to_icu  icu_los_days  n_chart_events  \
0     EMERGENCY ROOM              0      0.410266             473   
1     EMERGENCY ROOM              0      3.893252            3836   
2     EMERGENCY ROOM              0      0.497535             446   
3     EMERGENCY ROOM              0      3.891447            3184   